# new_run — spec → resolve → preflight → launch

Describe **only the target**. The resolver backward-chains through `requires()` to
enumerate every job, dedupes by content-addressed uid, and writes
`runs/{run_id}/config.json` with each job's params, dependency uids, and the
**actual files** it needs. Preflight re-instantiates every class from the config
(pydantic validates) and checks it matches the code before anything spawns.

Part 1 is pretraining. Part 2 stacks SFT → reward model → RL → eval on top,
reusing every pretraining artifact from cache.

In [1]:
from pathlib import Path
import json, uuid

from jobs import (
    SourceJob,
    TokenizerTrainJob,
    TokenizeJob,
    TrainJob,
    DataJob,
    SFTJob,
    RewardModelJob,
    RLJob,
    EvalJob,
    resolve,
    preflight,
    run_job,
)

VOLUME = Path("volume")  # stand-in for the Modal volume mount
VOLUME.mkdir(exist_ok=True)

## Part 1 — pretraining spec

In [2]:
VOCAB_SIZE = 64

TRAIN_SET = ["odyssey", "mobydick"]
VALID_SET = ["montecristo", "romeojuliet"]
TOKEN_SET = ["odyssey"]


def src(name):  # expand a name into SourceJob params
    return {"name": name, "normalizer": "lowercase"}


TOKENIZER = {
    "kind": "bpe",
    "vocab_size": VOCAB_SIZE,
    "special_tokens": ["<|endoftext|>", "<|begin|>", "<|end|>"],
    "fit_sources": [src(n) for n in TOKEN_SET],
}

PRETRAIN = {
    "model": {"n_layer": 2, "n_embd": 64, "lr": 3e-4, "epochs": 1},
    "tokenizer": TOKENIZER,
    "train_sources": [src(n) for n in TRAIN_SET],
    "val_sources": [src(n) for n in VALID_SET],
}

target = TrainJob(PRETRAIN)
target.uid

'train-4d63ffa20dbf'

### Resolve

`odyssey` is in both `TOKEN_SET` and `TRAIN_SET` but appears as **one** SourceJob,
feeding both the tokenizer fit and its own tokenize job.

In [3]:
config = resolve(target, VOLUME)
for uid, e in config["jobs"].items():
    print(
        f"{uid:28s} deps={len(e['dependencies'])}  inputs={len(e['input_files'])}  cached={e['cached']}"
    )
print(f"\ntarget: {config['target']}   total jobs: {len(config['jobs'])}")

source-aa33ec189fbe          deps=0  inputs=0  cached=False
tok_train-d19e3acefe70       deps=1  inputs=1  cached=False
tokenize-47d0061511ad        deps=2  inputs=2  cached=False
source-750b8c63fdb7          deps=0  inputs=0  cached=False
tokenize-cb0c232094b6        deps=2  inputs=2  cached=False
source-77f9924a0c02          deps=0  inputs=0  cached=False
tokenize-babdd3206979        deps=2  inputs=2  cached=False
source-7cd5ab6f4014          deps=0  inputs=0  cached=False
tokenize-c813264e89f8        deps=2  inputs=2  cached=False
train-4d63ffa20dbf           deps=4  inputs=4  cached=False

target: train-4d63ffa20dbf   total jobs: 10


In [4]:
print(json.dumps(config["jobs"][config["target"]], indent=2))

{
  "job_type": "train",
  "code_version": "1",
  "parameters": {
    "model": {
      "n_layer": 2,
      "n_embd": 64,
      "lr": 0.0003,
      "epochs": 1
    },
    "tokenizer": {
      "kind": "bpe",
      "vocab_size": 64,
      "special_tokens": [
        "<|endoftext|>",
        "<|begin|>",
        "<|end|>"
      ],
      "fit_sources": [
        {
          "name": "odyssey",
          "normalizer": "lowercase"
        }
      ]
    },
    "train_sources": [
      {
        "name": "odyssey",
        "normalizer": "lowercase"
      },
      {
        "name": "mobydick",
        "normalizer": "lowercase"
      }
    ],
    "val_sources": [
      {
        "name": "montecristo",
        "normalizer": "lowercase"
      },
      {
        "name": "romeojuliet",
        "normalizer": "lowercase"
      }
    ]
  },
  "dependencies": [
    "tokenize-47d0061511ad",
    "tokenize-babdd3206979",
    "tokenize-c813264e89f8",
    "tokenize-cb0c232094b6"
  ],
  "input_files": [
    "dat

### Save the run, preflight, launch

In [5]:
run_id = uuid.uuid4().hex[:8]
run_dir = VOLUME / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "config.json").write_text(json.dumps(config, indent=2))

report = preflight(config, VOLUME)
print("config valid:", report["ok"], "\n")
for uid, r in report["jobs"].items():
    state = "CACHED" if r["cached"] else ("READY" if r["ready"] else "BLOCKED")
    print(f"{uid:28s} {state:8s} missing_inputs={len(r['missing_inputs'])}")

config valid: True 

source-aa33ec189fbe          READY    missing_inputs=0
tok_train-d19e3acefe70       BLOCKED  missing_inputs=1
tokenize-47d0061511ad        BLOCKED  missing_inputs=2
source-750b8c63fdb7          READY    missing_inputs=0
tokenize-cb0c232094b6        BLOCKED  missing_inputs=2
source-77f9924a0c02          READY    missing_inputs=0
tokenize-babdd3206979        BLOCKED  missing_inputs=2
source-7cd5ab6f4014          READY    missing_inputs=0
tokenize-c813264e89f8        BLOCKED  missing_inputs=2
train-4d63ffa20dbf           BLOCKED  missing_inputs=4


In [6]:
try:  # the no-GPU-before-deps gate
    run_job(VOLUME, config, config["target"])
except RuntimeError as e:
    print("refused:", e)

refused: train-4d63ffa20dbf: BLOCKED, missing inputs: ['data/tokenizers/tok_train-d19e3acefe70/bin/source-750b8c63fdb7.bin', 'data/tokenizers/tok_train-d19e3acefe70/bin/source-77f9924a0c02.bin', 'data/tokenizers/tok_train-d19e3acefe70/bin/source-7cd5ab6f4014.bin', 'data/tokenizers/tok_train-d19e3acefe70/bin/source-aa33ec189fbe.bin']


In [7]:
for uid in config["jobs"]:  # topo order from the resolver
    print(run_job(VOLUME, config, uid))

source-aa33ec189fbe: done -> ['data/sources/source-aa33ec189fbe/content.txt']
tok_train-d19e3acefe70: done -> ['data/tokenizers/tok_train-d19e3acefe70/tokenizer.obj']
tokenize-47d0061511ad: done -> ['data/tokenizers/tok_train-d19e3acefe70/bin/source-aa33ec189fbe.bin']
source-750b8c63fdb7: done -> ['data/sources/source-750b8c63fdb7/content.txt']
tokenize-cb0c232094b6: done -> ['data/tokenizers/tok_train-d19e3acefe70/bin/source-750b8c63fdb7.bin']
source-77f9924a0c02: done -> ['data/sources/source-77f9924a0c02/content.txt']
tokenize-babdd3206979: done -> ['data/tokenizers/tok_train-d19e3acefe70/bin/source-77f9924a0c02.bin']
source-7cd5ab6f4014: done -> ['data/sources/source-7cd5ab6f4014/content.txt']
tokenize-c813264e89f8: done -> ['data/tokenizers/tok_train-d19e3acefe70/bin/source-7cd5ab6f4014.bin']
train-4d63ffa20dbf: done -> ['data/checkpoints/train-4d63ffa20dbf/model.pt']


## Part 2 — post-training: SFT → reward model → RL → eval

Each stage takes the **params** of the stage below it, not a uid — so the graph
composes declaratively and identity propagates through the Merkle hash. The RL
rollout/score/update loop lives inside `RLJob.run()`; the DAG only sees params
in, policy out.

In [8]:
SFT = {
    "base": PRETRAIN,  # the whole pretraining spec, by value
    "tokenizer": TOKENIZER,
    "dataset": {"name": "chat_sft"},
    "lr": 1e-5,
    "epochs": 2,
}

REWARD_MODEL = {
    "base": SFT,
    "dataset": {"name": "chat_prefs"},  # preference pairs
    "lr": 1e-4,
}

RL = {
    "policy": SFT,  # init policy = the SFT checkpoint
    "reward_model": REWARD_MODEL,
    "prompts": {"name": "chat_prompts"},
    "algo": "ppo",
    "steps": 4,
    "kl_coef": 0.1,
}

post_target = EvalJob(
    {"model_job_type": "rl", "model": RL, "benchmark": "toy_perplexity"}
)
post_target.uid

'eval-8b80b20d6b49'

### Resolve the post-training graph

Everything from Part 1 comes back **cached** — the pretrained checkpoint is
content-addressed, so SFT reuses it across runs instead of retraining.

In [9]:
post_config = resolve(post_target, VOLUME)
for uid, e in post_config["jobs"].items():
    tag = "cached" if e["cached"] else "TO RUN"
    print(f"{uid:28s} {tag:7s} deps={len(e['dependencies'])}")
print(
    f"\ntotal: {len(post_config['jobs'])} jobs, "
    f"{sum(e['cached'] for e in post_config['jobs'].values())} already done"
)

source-aa33ec189fbe          cached  deps=0
tok_train-d19e3acefe70       cached  deps=1
tokenize-47d0061511ad        cached  deps=2
source-750b8c63fdb7          cached  deps=0
tokenize-cb0c232094b6        cached  deps=2
source-77f9924a0c02          cached  deps=0
tokenize-babdd3206979        cached  deps=2
source-7cd5ab6f4014          cached  deps=0
tokenize-c813264e89f8        cached  deps=2
train-4d63ffa20dbf           cached  deps=4
data-7ab65c702349            TO RUN  deps=0
sft-0c833d8e4367             TO RUN  deps=3
data-44c9b2fd2d1a            TO RUN  deps=0
rm-c5d40dd7a2b1              TO RUN  deps=3
data-6677ec6db2db            TO RUN  deps=0
rl-880f42c7d934              TO RUN  deps=3
eval-8b80b20d6b49            TO RUN  deps=1

total: 17 jobs, 10 already done


In [10]:
post_run_id = uuid.uuid4().hex[:8]
d = VOLUME / "runs" / post_run_id
d.mkdir(parents=True, exist_ok=True)
(d / "config.json").write_text(json.dumps(post_config, indent=2))

print("preflight ok:", preflight(post_config, VOLUME)["ok"])
for uid in post_config["jobs"]:
    print(run_job(VOLUME, post_config, uid))

preflight ok: True
source-aa33ec189fbe: cached, skipping
tok_train-d19e3acefe70: cached, skipping
tokenize-47d0061511ad: cached, skipping
source-750b8c63fdb7: cached, skipping
tokenize-cb0c232094b6: cached, skipping
source-77f9924a0c02: cached, skipping
tokenize-babdd3206979: cached, skipping
source-7cd5ab6f4014: cached, skipping
tokenize-c813264e89f8: cached, skipping
train-4d63ffa20dbf: cached, skipping
data-7ab65c702349: done -> ['data/datasets/data-7ab65c702349/rows.json']
sft-0c833d8e4367: done -> ['data/checkpoints/sft-0c833d8e4367/model.pt']
data-44c9b2fd2d1a: done -> ['data/datasets/data-44c9b2fd2d1a/rows.json']
rm-c5d40dd7a2b1: done -> ['data/reward_models/rm-c5d40dd7a2b1/rm.pt']
data-6677ec6db2db: done -> ['data/datasets/data-6677ec6db2db/rows.json']
rl-880f42c7d934: done -> ['data/checkpoints/rl-880f42c7d934/policy.pt', 'data/checkpoints/rl-880f42c7d934/rl_metrics.json']
eval-8b80b20d6b49: done -> ['data/evals/eval-8b80b20d6b49/metrics.json']


### Results — lineage is readable straight off the artifacts

In [11]:
rl_uid = next(u for u in post_config["jobs"] if u.startswith("rl-"))
metrics = json.loads((VOLUME / post_config["jobs"][rl_uid]["outputs"][1]).read_text())
print(json.dumps(metrics, indent=2))

print(
    json.dumps(
        json.loads(
            (VOLUME / post_config["jobs"][post_target.uid]["outputs"][0]).read_text()
        ),
        indent=2,
    )
)

{
  "algo": "ppo",
  "steps": 4,
  "n_prompts": 3,
  "history": [
    {
      "step": 0,
      "mean_reward": 2e-05
    },
    {
      "step": 1,
      "mean_reward": 2e-05
    },
    {
      "step": 2,
      "mean_reward": 2e-05
    },
    {
      "step": 3,
      "mean_reward": 2e-05
    }
  ]
}
{
  "benchmark": "toy_perplexity",
  "model_uid": "rl-880f42c7d934",
  "stage": "rl-ppo",
  "vocab_covered": 11,
  "mean_score": 0.103668
}


### Fan-out: eval every stage, one job per (model, benchmark)

Adding evals costs nothing — the models are already built, so only the new
EvalJobs run.

In [12]:
stages = {"train": PRETRAIN, "sft": SFT, "rl": RL}
for stage, params in stages.items():
    ev = EvalJob({"model_job_type": stage, "model": params})
    cfg = resolve(ev, VOLUME)
    for uid in cfg["jobs"]:
        run_job(VOLUME, cfg, uid)
    m = json.loads((VOLUME / cfg["jobs"][ev.uid]["outputs"][0]).read_text())
    print(f"{stage:6s} -> mean_score={m['mean_score']:<10} vocab={m['vocab_covered']}")

train  -> mean_score=0.090909   vocab=11


sft    -> mean_score=0.103637   vocab=11
rl     -> mean_score=0.103668   vocab=11


### Preflight catches drift

Tamper with a param (stale or hand-edited config): the recomputed uid no longer
matches its key, and the run is rejected before any container spawns.

In [13]:
bad = json.loads(json.dumps(post_config))
first = next(iter(bad["jobs"]))
bad["jobs"][first]["parameters"]["name"] = "hacked"

r = preflight(bad, VOLUME)
print("config valid:", r["ok"])
print(r["jobs"][first]["issues"])

config valid: False
['uid mismatch: config=source-aa33ec189fbe class=source-76ce4d92c817', "outputs don't match class definition"]


In [14]:
for p in sorted(VOLUME.rglob("*")):
    if p.is_file():
        print(p.relative_to(VOLUME))

data/checkpoints/rl-880f42c7d934/policy.pt
data/checkpoints/rl-880f42c7d934/rl_metrics.json
data/checkpoints/sft-0c833d8e4367/model.pt
data/checkpoints/train-4d63ffa20dbf/model.pt
data/datasets/data-44c9b2fd2d1a/rows.json
data/datasets/data-6677ec6db2db/rows.json
data/datasets/data-7ab65c702349/rows.json
data/evals/eval-0f5c654e491d/metrics.json
data/evals/eval-8b80b20d6b49/metrics.json
data/evals/eval-a53280ff540c/metrics.json
data/reward_models/rm-c5d40dd7a2b1/rm.pt
data/sources/source-750b8c63fdb7/content.txt
data/sources/source-77f9924a0c02/content.txt
data/sources/source-7cd5ab6f4014/content.txt
data/sources/source-aa33ec189fbe/content.txt
data/tokenizers/tok_train-d19e3acefe70/bin/source-750b8c63fdb7.bin
data/tokenizers/tok_train-d19e3acefe70/bin/source-77f9924a0c02.bin
data/tokenizers/tok_train-d19e3acefe70/bin/source-7cd5ab6f4014.bin
data/tokenizers/tok_train-d19e3acefe70/bin/source-aa33ec189fbe.bin
data/tokenizers/tok_train-d19e3acefe70/tokenizer.obj
runs/7cab812f/config.json
